# Large Language Models: Architecture Deep Dive

**Module 08: Large Language Models**  
**Objective**: Understand LLM architecture, scaling laws, and emergent abilities

LLMs have revolutionized AI:
- **GPT-3**: 175B parameters, few-shot learning
- **GPT-4**: Multimodal, reasoning capabilities
- **LLaMA**: Open-source, efficient
- **Claude**: Long context (100K+ tokens)

## What You'll Learn
1. GPT architecture in detail
2. Scaling laws (Chinchilla, Kaplan)
3. Tokenization (BPE, WordPiece, SentencePiece)
4. KV caching for efficient inference
5. Positional encoding variants (RoPE, ALiBi)
6. Emergent abilities and in-context learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, Tuple
import math

np.random.seed(42)

## 1. GPT Architecture Review

**GPT = Generative Pre-trained Transformer**

**Key components**:
- Decoder-only architecture (causal self-attention)
- Token + positional embeddings
- Multi-head self-attention
- Feed-forward network
- Layer normalization
- Autoregressive generation

**Training objective**: Predict next token given previous tokens

$$P(x_{t} | x_{<t}) = \text{softmax}(W_e h_t)$$

In [ ]:
class GPTBlock:
    """Single GPT transformer block."""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff
        self.d_k = d_model // n_heads
        
        # Layer norm
        self.ln1_gamma = np.ones(d_model)
        self.ln1_beta = np.zeros(d_model)
        self.ln2_gamma = np.ones(d_model)
        self.ln2_beta = np.zeros(d_model)
        
        # Attention
        self.W_qkv = np.random.randn(d_model, 3 * d_model) * 0.02
        self.W_o = np.random.randn(d_model, d_model) * 0.02
        
        # FFN
        self.W_ff1 = np.random.randn(d_model, d_ff) * 0.02
        self.b_ff1 = np.zeros(d_ff)
        self.W_ff2 = np.random.randn(d_ff, d_model) * 0.02
        self.b_ff2 = np.zeros(d_model)
    
    def layer_norm(self, x, gamma, beta, eps=1e-5):
        """Layer normalization."""
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        return gamma * (x - mean) / np.sqrt(var + eps) + beta
    
    def gelu(self, x):
        """GELU activation (used in GPT)."""
        return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
    
    def causal_self_attention(self, x):
        """Causal (masked) self-attention."""
        batch_size, seq_len, d_model = x.shape
        
        # Compute Q, K, V
        qkv = x @ self.W_qkv  # (batch, seq_len, 3*d_model)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.n_heads, self.d_k)
        qkv = qkv.transpose(2, 0, 3, 1, 4)  # (3, batch, heads, seq_len, d_k)
        
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Attention scores
        scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        
        # Causal mask
        mask = np.triu(np.ones((seq_len, seq_len)) * -1e10, k=1)
        scores = scores + mask
        
        # Softmax
        attn_weights = self._softmax(scores)
        
        # Apply to values
        out = attn_weights @ v  # (batch, heads, seq_len, d_k)
        
        # Concatenate heads
        out = out.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, d_model)
        
        # Output projection
        out = out @ self.W_o
        
        return out
    
    def _softmax(self, x):
        """Numerically stable softmax."""
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)
    
    def forward(self, x):
        """Forward pass through GPT block."""
        # Pre-norm + attention + residual
        x_norm = self.layer_norm(x, self.ln1_gamma, self.ln1_beta)
        attn_out = self.causal_self_attention(x_norm)
        x = x + attn_out
        
        # Pre-norm + FFN + residual
        x_norm = self.layer_norm(x, self.ln2_gamma, self.ln2_beta)
        ff_out = x_norm @ self.W_ff1 + self.b_ff1
        ff_out = self.gelu(ff_out)
        ff_out = ff_out @ self.W_ff2 + self.b_ff2
        x = x + ff_out
        
        return x

# Test GPT block
print("Testing GPT Block\n")

block = GPTBlock(d_model=768, n_heads=12, d_ff=3072)
x = np.random.randn(2, 10, 768)
output = block.forward(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"d_model: 768, n_heads: 12, d_k: {block.d_k}")
print("\n✓ GPT block maintains shape through residual connections!")

## 2. Scaling Laws

**Kaplan et al. (2020)**: Performance scales as power law with model size, dataset size, compute

$$L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}$$

where:
- $L$: Loss
- $N$: Number of parameters
- $N_c$: Critical parameter count
- $\alpha_N \approx 0.076$

**Chinchilla (2022)**: Compute-optimal training
- For compute budget $C$, optimal: $N \propto C^{0.5}$, $D \propto C^{0.5}$
- Previous models were under-trained!

In [ ]:
def scaling_law_loss(N, Nc=8.8e13, alpha=0.076):
    """
    Compute loss according to scaling law.
    
    Args:
        N: number of parameters
        Nc: critical parameter count
        alpha: scaling exponent
    
    Returns:
        loss
    """
    return (Nc / N) ** alpha

# Plot scaling laws
param_counts = np.logspace(6, 12, 100)  # 1M to 1T parameters
losses = [scaling_law_loss(N) for N in param_counts]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.loglog(param_counts, losses, linewidth=2)
plt.xlabel('Number of Parameters')
plt.ylabel('Loss')
plt.title('Scaling Law: Loss vs Model Size')
plt.grid(True, alpha=0.3)

# Mark notable models
models = {
    'GPT-2': (1.5e9, scaling_law_loss(1.5e9)),
    'GPT-3': (175e9, scaling_law_loss(175e9)),
    'GPT-4 (est)': (1.5e12, scaling_law_loss(1.5e12))
}

for name, (params, loss) in models.items():
    plt.scatter([params], [loss], s=100, zorder=5)
    plt.annotate(name, (params, loss), xytext=(10, 10), 
                textcoords='offset points', fontsize=9)

# Chinchilla optimal allocation
compute_budgets = np.logspace(18, 25, 100)  # FLOPs
optimal_params = (compute_budgets / 6) ** 0.5 / 1e9  # Approximate
optimal_tokens = (compute_budgets / 6) ** 0.5 / 1e9

plt.subplot(1, 2, 2)
plt.loglog(compute_budgets, optimal_params, label='Optimal Parameters (B)', linewidth=2)
plt.loglog(compute_budgets, optimal_tokens, label='Optimal Tokens (B)', linewidth=2, linestyle='--')
plt.xlabel('Compute Budget (FLOPs)')
plt.ylabel('Count (Billions)')
plt.title('Chinchilla: Compute-Optimal Scaling')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Insights from Scaling Laws:")
print("1. Loss decreases as power law with model size")
print("2. Larger models are more sample-efficient")
print("3. Compute-optimal: Balance model size and training data")
print("4. GPT-3 was under-trained relative to its size")

## 3. Tokenization

**Subword tokenization**: Balance vocabulary size and sequence length

**Popular methods**:
- **BPE (Byte-Pair Encoding)**: GPT, used by OpenAI
- **WordPiece**: BERT, used by Google
- **SentencePiece**: Universal, language-agnostic

**Benefits**:
- Fixed vocabulary size (typically 32K-100K)
- Handle rare words via subwords
- Language-agnostic (works on bytes)

In [ ]:
class SimpleBPE:
    """Simplified Byte-Pair Encoding."""
    
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = []
    
    def get_stats(self, words):
        """Count frequency of adjacent pairs."""
        pairs = {}
        for word, freq in words.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pair = (symbols[i], symbols[i+1])
                pairs[pair] = pairs.get(pair, 0) + freq
        return pairs
    
    def merge_vocab(self, pair, words):
        """Merge most frequent pair."""
        new_words = {}
        bigram = ' '.join(pair)
        replacement = ''.join(pair)
        
        for word in words:
            new_word = word.replace(bigram, replacement)
            new_words[new_word] = words[word]
        
        return new_words
    
    def train(self, corpus, num_merges=100):
        """Train BPE on corpus."""
        # Initialize with characters
        words = {}
        for word in corpus.split():
            word = ' '.join(list(word)) + ' </w>'
            words[word] = words.get(word, 0) + 1
        
        # Perform merges
        for i in range(num_merges):
            pairs = self.get_stats(words)
            if not pairs:
                break
            
            best_pair = max(pairs, key=pairs.get)
            words = self.merge_vocab(best_pair, words)
            self.merges.append(best_pair)
            
            if (i + 1) % 20 == 0:
                print(f"Merge {i+1}: {best_pair} -> {''.join(best_pair)}")
        
        # Build vocabulary
        self.vocab = set()
        for word in words:
            self.vocab.update(word.split())
        
        return words
    
    def tokenize(self, text):
        """Tokenize text using learned merges."""
        # Start with character-level
        tokens = ' '.join(list(text)) + ' </w>'
        
        # Apply merges in order
        for pair in self.merges:
            bigram = ' '.join(pair)
            replacement = ''.join(pair)
            tokens = tokens.replace(bigram, replacement)
        
        return tokens.split()

# Test BPE
print("\nTesting Byte-Pair Encoding\n")

corpus = "low lower lowest lower low lowest low lower"
bpe = SimpleBPE()
vocab = bpe.train(corpus, num_merges=10)

print(f"\nFinal vocabulary size: {len(bpe.vocab)}")
print(f"Vocabulary: {sorted(bpe.vocab)[:20]}")

# Tokenize new text
test_text = "lowest"
tokens = bpe.tokenize(test_text)
print(f"\nTokenization of '{test_text}': {tokens}")

## 4. KV Caching for Efficient Inference

**Problem**: Autoregressive generation recomputes attention for all previous tokens

**Solution**: Cache key and value tensors

**Savings**: $O(n^2) \rightarrow O(n)$ for generating $n$ tokens

$$\text{Attention}(Q_t, [K_{<t}, K_t], [V_{<t}, V_t])$$

In [ ]:
class AttentionWithKVCache:
    """Attention with KV caching."""
    
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = np.random.randn(d_model, d_model) * 0.02
        self.W_k = np.random.randn(d_model, d_model) * 0.02
        self.W_v = np.random.randn(d_model, d_model) * 0.02
        self.W_o = np.random.randn(d_model, d_model) * 0.02
        
        # Cache
        self.k_cache = None
        self.v_cache = None
    
    def forward(self, x, use_cache=False, past_kv=None):
        """
        Forward with optional KV caching.
        
        Args:
            x: (batch, seq_len, d_model) - can be single token or full sequence
            use_cache: whether to use/update cache
            past_kv: tuple of (k_cache, v_cache) from previous step
        
        Returns:
            output, (k_cache, v_cache)
        """
        batch_size, seq_len, _ = x.shape
        
        # Compute Q, K, V for current input
        q = (x @ self.W_q).reshape(batch_size, seq_len, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        k = (x @ self.W_k).reshape(batch_size, seq_len, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        v = (x @ self.W_v).reshape(batch_size, seq_len, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        
        # Use cached K, V if available
        if use_cache and past_kv is not None:
            k_cache, v_cache = past_kv
            k = np.concatenate([k_cache, k], axis=2)  # Concatenate along sequence dimension
            v = np.concatenate([v_cache, v], axis=2)
        
        # Attention
        scores = (q @ k.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        
        # Causal mask (only for current positions)
        full_seq_len = k.shape[2]
        if seq_len == 1:
            # Single token generation, no masking needed
            pass
        else:
            # Full sequence, apply causal mask
            mask = np.triu(np.ones((seq_len, full_seq_len)) * -1e10, k=full_seq_len - seq_len + 1)
            scores = scores + mask
        
        attn_weights = self._softmax(scores)
        out = attn_weights @ v
        
        # Reshape and project
        out = out.transpose(0, 2, 1, 3).reshape(batch_size, seq_len, self.d_model)
        out = out @ self.W_o
        
        # Update cache
        new_kv = (k, v) if use_cache else None
        
        return out, new_kv
    
    def _softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

# Compare with and without caching
print("\nKV Caching Demonstration\n")

attn = AttentionWithKVCache(d_model=512, n_heads=8)

# Simulate generation of 10 tokens
batch_size = 1
d_model = 512

# Method 1: No caching (recompute everything)
print("Without KV caching:")
total_ops_no_cache = 0
for t in range(1, 11):
    x = np.random.randn(batch_size, t, d_model)  # Full sequence
    out, _ = attn.forward(x, use_cache=False)
    ops = t * t  # Attention is O(n^2)
    total_ops_no_cache += ops

print(f"Total operations: {total_ops_no_cache}")

# Method 2: With caching
print("\nWith KV caching:")
total_ops_cache = 0
past_kv = None
for t in range(1, 11):
    x_t = np.random.randn(batch_size, 1, d_model)  # Single token
    out, past_kv = attn.forward(x_t, use_cache=True, past_kv=past_kv)
    ops = t  # Only compute attention for new token
    total_ops_cache += ops

print(f"Total operations: {total_ops_cache}")
print(f"\nSpeedup: {total_ops_no_cache / total_ops_cache:.1f}x")
print("✓ KV caching dramatically reduces computation for long sequences!")

## 5. Positional Encoding Variants

**Standard sinusoidal**: Works but doesn't extrapolate well to longer sequences

**Improvements**:
- **RoPE (Rotary Position Embedding)**: Used in LLaMA, PaLM
- **ALiBi (Attention with Linear Biases)**: Used in BLOOM

**RoPE**: Apply rotation to Q and K based on position

$$\text{RoPE}(x, m) = x \cdot e^{im\theta}$$

In [ ]:
def rotary_position_embedding(seq_len, d_model):
    """
    Compute RoPE embeddings.
    
    Args:
        seq_len: sequence length
        d_model: must be even
    
    Returns:
        cos, sin: (seq_len, d_model//2) for rotation
    """
    assert d_model % 2 == 0
    
    # Position indices
    position = np.arange(seq_len)[:, np.newaxis]
    
    # Dimension indices (pairs)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    # Compute angles
    angles = position * div_term  # (seq_len, d_model//2)
    
    cos = np.cos(angles)
    sin = np.sin(angles)
    
    return cos, sin

def apply_rotary_embedding(x, cos, sin):
    """
    Apply RoPE rotation.
    
    Args:
        x: (batch, seq_len, d_model)
        cos, sin: (seq_len, d_model//2)
    
    Returns:
        rotated x
    """
    batch_size, seq_len, d_model = x.shape
    
    # Split into pairs
    x1 = x[..., 0::2]  # Even dimensions
    x2 = x[..., 1::2]  # Odd dimensions
    
    # Rotate
    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos
    
    # Interleave back
    result = np.empty_like(x)
    result[..., 0::2] = rotated_x1
    result[..., 1::2] = rotated_x2
    
    return result

# Test RoPE
seq_len = 20
d_model = 64

cos, sin = rotary_position_embedding(seq_len, d_model)

print(f"\nRoPE Embeddings")
print(f"Sequence length: {seq_len}")
print(f"Model dimension: {d_model}")
print(f"Cos shape: {cos.shape}")
print(f"Sin shape: {sin.shape}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cos.T, cmap='coolwarm', center=0, ax=axes[0], cbar=True)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('RoPE: Cosine Components')

sns.heatmap(sin.T, cmap='coolwarm', center=0, ax=axes[1], cbar=True)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Dimension')
axes[1].set_title('RoPE: Sine Components')

plt.tight_layout()
plt.show()

print("\n✓ RoPE enables better length extrapolation!")

## 6. Emergent Abilities

**Definition**: Abilities that appear suddenly at scale, not present in smaller models

**Examples**:
1. **Few-shot learning**: Learn from examples in prompt
2. **Chain-of-thought reasoning**: Break down complex problems
3. **Instruction following**: Understand and follow instructions
4. **Code generation**: Write functional code
5. **Multilingual translation**: Without explicit training

**Key insight**: Larger models exhibit qualitatively different behaviors

In [ ]:
def simulate_emergent_abilities():
    """Visualize emergent abilities at scale."""
    
    # Simulate performance on various tasks
    model_sizes = [0.1, 0.5, 1, 6, 13, 30, 70, 175]  # Billions of parameters
    
    # Different task types
    basic_tasks = [20, 40, 60, 75, 82, 87, 90, 92]  # Gradual improvement
    emergent_tasks = [5, 8, 10, 15, 20, 65, 85, 90]  # Sudden jump
    reasoning_tasks = [2, 5, 8, 12, 18, 25, 70, 88]  # Another emergent pattern
    
    plt.figure(figsize=(12, 6))
    
    plt.plot(model_sizes, basic_tasks, marker='o', linewidth=2, label='Basic Tasks (e.g., sentiment)')
    plt.plot(model_sizes, emergent_tasks, marker='s', linewidth=2, label='Emergent: Few-shot learning')
    plt.plot(model_sizes, reasoning_tasks, marker='^', linewidth=2, label='Emergent: Chain-of-thought')
    
    plt.axvline(x=13, color='red', linestyle='--', alpha=0.3, label='Critical scale')
    
    plt.xlabel('Model Size (Billions of Parameters)')
    plt.ylabel('Performance (%)')
    plt.title('Emergent Abilities in Large Language Models')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xscale('log')
    
    plt.tight_layout()
    plt.show()
    
    print("\nEmergent Abilities:")
    print("1. Small models (<10B): Basic pattern matching")
    print("2. Medium models (10-30B): Some reasoning, inconsistent")
    print("3. Large models (>30B): Complex reasoning, few-shot learning")
    print("\n✓ Qualitative phase transitions at scale!")

simulate_emergent_abilities()

## 7. Model Architecture Comparison

In [ ]:
# Compare different LLM architectures
architectures = {
    'GPT-3 (175B)': {
        'params': 175e9,
        'layers': 96,
        'd_model': 12288,
        'heads': 96,
        'd_ff': 49152,
        'context': 2048,
        'position': 'Learned',
        'activation': 'GELU',
        'year': 2020
    },
    'LLaMA-70B': {
        'params': 70e9,
        'layers': 80,
        'd_model': 8192,
        'heads': 64,
        'd_ff': 28672,
        'context': 2048,
        'position': 'RoPE',
        'activation': 'SwiGLU',
        'year': 2023
    },
    'GPT-4 (est)': {
        'params': 1.5e12,
        'layers': 120,
        'd_model': 16384,
        'heads': 128,
        'd_ff': 65536,
        'context': 8192,
        'position': 'RoPE',
        'activation': 'GELU',
        'year': 2023
    },
    'Claude-2': {
        'params': 200e9,
        'layers': 100,
        'd_model': 14336,
        'heads': 112,
        'd_ff': 57344,
        'context': 100000,
        'position': 'RoPE',
        'activation': 'GELU',
        'year': 2023
    }
}

print("\nLarge Language Model Architecture Comparison\n")
print("="*110)
print(f"{'Model':<18} {'Params':<12} {'Layers':<8} {'d_model':<10} {'Heads':<8} {'Context':<10} {'Position':<10}")
print("="*110)

for name, specs in architectures.items():
    params_str = f"{specs['params']/1e9:.0f}B" if specs['params'] < 1e12 else f"{specs['params']/1e12:.1f}T"
    print(f"{name:<18} {params_str:<12} {specs['layers']:<8} {specs['d_model']:<10} {specs['heads']:<8} "
          f"{specs['context']:<10} {specs['position']:<10}")

print("="*110)

# Visualize parameter distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model sizes
models = list(architectures.keys())
params = [architectures[m]['params']/1e9 for m in models]
colors = ['skyblue', 'lightcoral', 'lightgreen', 'lightyellow']

axes[0].barh(models, params, color=colors)
axes[0].set_xlabel('Parameters (Billions)')
axes[0].set_title('Model Size Comparison')
axes[0].set_xscale('log')
axes[0].grid(axis='x', alpha=0.3)

# Context length
contexts = [architectures[m]['context'] for m in models]
axes[1].barh(models, contexts, color=colors)
axes[1].set_xlabel('Context Length (tokens)')
axes[1].set_title('Context Window Comparison')
axes[1].set_xscale('log')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

You've mastered LLM architecture fundamentals:
- ✅ GPT architecture in detail
- ✅ Scaling laws (Kaplan, Chinchilla)
- ✅ Tokenization (BPE)
- ✅ KV caching for efficient inference
- ✅ Positional encoding variants (RoPE)
- ✅ Emergent abilities at scale

**Key insights**:
1. **Scaling**: Larger models show power-law improvements
2. **Compute-optimal**: Balance model size and training data
3. **Architecture choices**: RoPE, SwiGLU, LayerNorm variations matter
4. **Efficiency**: KV caching, Flash Attention for speed
5. **Emergence**: New abilities appear at scale (~10-30B+ parameters)

**Training challenges**:
- Compute cost: $1M-$100M+ for large models
- Data quality: Clean, diverse corpus crucial
- Hyperparameters: Learning rate, batch size, warmup
- Stability: Gradient clipping, mixed precision

**Next Notebook**: Implementing and using LLMs with PyTorch/HuggingFace

## Exercises

1. **Implement Flash Attention**: Memory-efficient attention for long sequences
2. **Scaling Law Analysis**: Fit power law to real model performance data
3. **Position Encoding Comparison**: Test RoPE vs ALiBi vs sinusoidal on extrapolation